In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import os


In [14]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=False, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [16]:
def train_model(model, optimizer, criterion, train_loader, device, num_epochs=5):
    model.to(device)
    model.train()
    for epoch in range(num_epochs):
        running_loss, correct = 0.0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct.double() / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model1 = models.vgg16(pretrained=True)
for param in model1.features.parameters():
    param.requires_grad = False

model1.classifier[6] = nn.Linear(4096, 10)

criterion = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.classifier.parameters(), lr=0.001)

print("\n--- Training Model 1 (Frozen Feature Extractor) ---")
train_model(model1, optimizer1, criterion, train_loader, device)


C:\Users\wayne\anaconda3\envs\GBC_Task\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\wayne\anaconda3\envs\GBC_Task\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\wayne/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100%|██████████| 528M/528M [00:09<00:00, 56.2MB/s] 



--- Training Model 1 (Frozen Feature Extractor) ---
Epoch 1/5 - Loss: 1.7010, Acc: 0.4249
Epoch 2/5 - Loss: 1.6159, Acc: 0.4469
Epoch 3/5 - Loss: 1.5890, Acc: 0.4551
Epoch 4/5 - Loss: 1.5697, Acc: 0.4621
Epoch 5/5 - Loss: 1.5447, Acc: 0.4686


In [19]:
model2 = models.vgg16(pretrained=True)
for name, param in model2.features.named_parameters():
    if int(name.split('.')[0]) >= 24:  # 解凍最後三個 conv blocks (5)
        param.requires_grad = True
    else:
        param.requires_grad = False

model2.classifier[6] = nn.Linear(4096, 10)

optimizer2 = optim.Adam(filter(lambda p: p.requires_grad, model2.parameters()), lr=0.0005)

print("\n--- Training Model 2 (Partial Fine-tuning) ---")
train_model(model2, optimizer2, criterion, train_loader, device)



--- Training Model 2 (Partial Fine-tuning) ---
Epoch 1/5 - Loss: 1.1060, Acc: 0.6264
Epoch 2/5 - Loss: 0.9088, Acc: 0.7017
Epoch 3/5 - Loss: 0.8233, Acc: 0.7259
Epoch 4/5 - Loss: 0.7955, Acc: 0.7373
Epoch 5/5 - Loss: 0.7610, Acc: 0.7499


In [20]:
model3 = models.vgg16(pretrained=True)

model3.classifier = nn.Sequential(
    nn.Linear(25088, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

for param in model3.parameters():
    param.requires_grad = True  # 全部解凍

optimizer3 = optim.Adam(model3.parameters(), lr=0.0001)

print("\n--- Training Model 3 (Full Fine-tuning + Custom Classifier + Augmentation) ---")
train_model(model3, optimizer3, criterion, train_loader, device)



--- Training Model 3 (Full Fine-tuning + Custom Classifier + Augmentation) ---
Epoch 1/5 - Loss: 0.7432, Acc: 0.7499
Epoch 2/5 - Loss: 0.4739, Acc: 0.8415
Epoch 3/5 - Loss: 0.3867, Acc: 0.8697
Epoch 4/5 - Loss: 0.3184, Acc: 0.8935
Epoch 5/5 - Loss: 0.2876, Acc: 0.9016


In [21]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc


In [22]:
print("Evaluating Model 1 (Frozen Feature Extractor)")
acc1 = evaluate_model(model1, test_loader, device)

print("Evaluating Model 2 (Partial Fine-tuning)")
acc2 = evaluate_model(model2, test_loader, device)

print("Evaluating Model 3 (Full Fine-tuning + Custom Classifier)")
acc3 = evaluate_model(model3, test_loader, device)


Evaluating Model 1 (Frozen Feature Extractor)
Test Accuracy: 0.5310
Evaluating Model 2 (Partial Fine-tuning)
Test Accuracy: 0.7741
Evaluating Model 3 (Full Fine-tuning + Custom Classifier)
Test Accuracy: 0.8851
